# Study 1: Imbalance Data Techniques for Heart Disease Prediction

 six imbalance-handling techniques using two baseline models:

- Logistic Regression (LR)
- Support Vector Machine (SVM)



In [ ]:
# ============================================================
# 1) Import libraries, configuration, and load dataset
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTEENN, SMOTETomek


TEST_SIZE = 0.30
TARGET_COLUMN = "HeartDisease_1"

# Place the dataset in the Data folder or update DATA_PATH.
DATA_PATH = "../Data/dataframeclass0_splits1.csv"


if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Please place the dataset in the Data folder or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=42,stratify=y,)

print(f"Dataset shape: {df.shape}")
print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Testing set:  X={X_test.shape}, y={y_test.shape}")
print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())
print("\nTesting class distribution:")
print(y_test.value_counts().sort_index())


In [ ]:
models = {
    "LR": LogisticRegression(random_state=42, max_iter=1000),
    "SVM": SVC(kernel="linear", C=1.0, random_state=42),
}

results = []


def class_distribution(y_values):
    """Return class distribution as a dictionary."""
    return pd.Series(y_values).value_counts().sort_index().to_dict()


def evaluate_model(model, X_train_resampled, y_train_resampled, X_test, y_test):
    """Train a model and return evaluation metrics on the unchanged test set."""
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1_score": f1_score(y_test, y_pred, zero_division=0),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }


def run_resampling_method(method_name, sampler):
    """
    Apply one imbalance-handling technique to the training set only,
    train LR and SVM, and store the evaluation results.
    """
    global results

    print(f"Running method: {method_name}")

    X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)

    print("Before resampling:", class_distribution(y_train))
    print("After resampling: ", class_distribution(y_resampled))

    for model_name, model in models.items():
        metrics = evaluate_model(
            model=model,
            X_train_resampled=X_resampled,
            y_train_resampled=y_resampled,
            X_test=X_test,
            y_test=y_test,
        )

        results.append({
            "Resampling_Method": method_name,
            "Model": model_name,
            "Train_Class_Distribution_Before": class_distribution(y_train),
            "Train_Class_Distribution_After": class_distribution(y_resampled),
            **metrics,
        })

    return pd.DataFrame(results).sort_values(
        by=["Recall", "F1_score"],
        ascending=False
    ).reset_index(drop=True)


# Under-sampling

In [ ]:
# ============================================================
# 1) Random Under Sampling (RUS)
# ============================================================

rus_sampler = RandomUnderSampler(random_state=42)

results_df = run_resampling_method(
    method_name="RUS",
    sampler=rus_sampler,
)

results_df


In [ ]:
# ============================================================
# 2) Tomek Links
# ============================================================

tomek_links_sampler = TomekLinks()

results_df = run_resampling_method(
    method_name="TomekLinks",
    sampler=tomek_links_sampler,
)

results_df


# OVER-Sampling

In [ ]:
# ============================================================
# 1) SMOTE
# ============================================================

smote_sampler = SMOTE(random_state=42)

results_df = run_resampling_method(
    method_name="SMOTE",
    sampler=smote_sampler,
)

results_df


In [ ]:
# ============================================================
# 2) ADASYN
# ============================================================

adasyn_sampler = ADASYN(random_state=42)

results_df = run_resampling_method(
    method_name="ADASYN",
    sampler=adasyn_sampler,
)

results_df


# Combination

In [ ]:
# ============================================================
# 1) SMOTEENN
# ============================================================

smoteenn_sampler = SMOTEENN(random_state=42)

results_df = run_resampling_method(
    method_name="SMOTEENN",
    sampler=smoteenn_sampler,
)

results_df


In [ ]:
# ============================================================
# 2) SMOTETomek
# ============================================================

smote_tomek_sampler = SMOTETomek(random_state=42)

results_df = run_resampling_method(
    method_name="SMOTETomek",
    sampler=smote_tomek_sampler,
)

results_df


In [ ]:
# ============================================================
# Save final summary results
# ============================================================

results_df = pd.DataFrame(results).sort_values(
    by=["Recall", "F1_score"],
    ascending=False
).reset_index(drop=True)

RESULTS_PATH = "study1_imbalance_lr_svm_results.csv"
results_df.to_csv(RESULTS_PATH, index=False)

print(f"Results saved to: {RESULTS_PATH}")
results_df
